# OULAD Student Risk: ES, LR, and SVM

This notebook compares three systems for binary at-risk student prediction using the processed OULAD dataset.

| Abbreviation | Meaning | Role |
|---|---|---|
| **Common:** | Shared dataset, features, and train/test split | Used by all systems |
| **DT:** | Decision Tree | Generates IF–THEN rules |
| **KB:** | Knowledge Base | Stores generated rules |
| **ES:** | Expert System | Applies KB rules using Experta |
| **LR:** | Logistic Regression | Machine Learning baseline |
| **SVM:** | Support Vector Machine | Machine Learning baseline |

`final_result` is converted into the binary target:

- `1 = At Risk` for `Fail` or `Withdrawn`
- `0 = Not At Risk` for `Pass` or `Distinction`

All systems use the same dataset, selected features, and train/test split.  
The ES uses unscaled features so the generated rules stay interpretable.  
LR and SVM use additional numeric scaling because they are scale-sensitive models.


## Common: Setup

Import libraries and define shared constants.


In [1]:
# Standard library imports
import json
import sys
import subprocess
import collections
import collections.abc
from pathlib import Path

# Data handling imports
import numpy as np
import pandas as pd

# Preprocessing and splitting imports
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# DT rule generation
from sklearn.tree import DecisionTreeClassifier, _tree

# ML baselines
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Fixed random seed for reproducibility
RANDOM_STATE = 42

# Display settings for clearer notebook outputs
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 140)


## Common: Load Dataset

Load the processed OULAD dataset created by `preprocess_oulad.R`.


In [2]:
# Processed OULAD dataset exported by the R preprocessing script.
DATA_PATH = Path("data/oulad_processed.csv")

# If the file is not in data/, also check beside the notebook.
if not DATA_PATH.exists():
    alternative_path = Path(DATA_PATH.name)

    if alternative_path.exists():
        DATA_PATH = alternative_path

# Stop execution with a clear message if the dataset is missing.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset file not found. Expected data/oulad_processed.csv. "
        "Run preprocess_oulad.R first."
    )

# The processed OULAD file is a regular comma-separated CSV.
df = pd.read_csv(DATA_PATH)

# Print a quick loading summary.
print(f"Loaded file: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Show the first rows to confirm successful loading.
df.head()


Loaded file: data\oulad_processed.csv
Dataset shape: 1452 rows, 11 columns


,date_registration,gender,region,highest_education,imd_band,age_band,studied_credits,disability,average_CA_score,total_clicks,final_result
0,-92,Female,Scotland,HE Qualification,40-50%,35-55,120,N,78.558140,474,Pass
1,-29,Female,West Midlands Region,A Level or Equivalent,20-30%,0-35,60,Y,87.255814,237,Pass
2,-29,Female,North Western Region,A Level or Equivalent,0-10%,35-55,60,N,76.116279,1929,Pass
3,-24,Female,Scotland,A Level or Equivalent,60-70%,35-55,60,N,83.488372,539,Distinction
4,-32,Female,South Region,A Level or Equivalent,90-100%,35-55,60,N,84.883721,238,Pass


## Common: Create Target and Features

Create the binary target from `final_result`.

The target mapping is:

- `Fail` and `Withdrawn` → `At Risk`
- `Pass` and `Distinction` → `Not At Risk`

`final_result` is removed from the input features after creating the target.


In [3]:
# These columns are required for this experiment.
required_columns = [
    "final_result",
    "date_registration",
    "average_CA_score",
    "total_clicks",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "studied_credits",
    "disability"
]

# Check whether any required column is missing.
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

# Raise an explicit error if the dataset does not match the expected structure.
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Copy the dataframe so the original loaded data remains unchanged.
data = df.copy()

# Convert the original OULAD final result into a binary target.
target_map = {
    "Fail": 1,
    "Withdrawn": 1,
    "Pass": 0,
    "Distinction": 0
}

unexpected_results = sorted(set(data["final_result"].dropna()) - set(target_map))

if unexpected_results:
    raise ValueError(f"Unexpected final_result values found: {unexpected_results}")

data["at_risk"] = data["final_result"].map(target_map)

# Human-readable class names used in reports.
class_names = ["Not At Risk", "At Risk"]

# Define feature groups explicitly so the modeling notebook follows the EDA decisions.
registration_features = [
    "date_registration"
]

academic_features = [
    "average_CA_score"
]

engagement_features = [
    "total_clicks"
]

categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "studied_credits",
    "disability"
]

numeric_features = (
    registration_features
    + academic_features
    + engagement_features
)

input_features = (
    numeric_features
    + categorical_features
)

# X_raw contains only the selected input features.
X_raw = data[input_features].copy()

# y contains the binary target labels.
y = data["at_risk"]

print(f"Input shape: {X_raw.shape}")
print("Target column: at_risk")
print("Removed from inputs: final_result")


Input shape: (1452, 10)
Target column: at_risk
Removed from inputs: final_result


## Common: Check Class Balance

Display the target distribution.


In [4]:
# Convert numeric target values to readable labels before counting.
target_distribution = (
    y.map({0: "Not At Risk", 1: "At Risk"})
    .value_counts()
    .to_frame("count")
)

# Add percentage of the full dataset for each class.
target_distribution["percent"] = (
    target_distribution["count"] / len(y) * 100
).round(2)

# Display class balance.
display(target_distribution)


,count,percent
at_risk,,
Not At Risk,959,66.05
At Risk,493,33.95


## Common: Review Feature Groups

Review the numeric and categorical feature groups used by all systems.


In [5]:
# Numeric features are used as continuous values.
print("Numeric Features:")
print(", ".join(numeric_features))

# Categorical features are one-hot encoded later.
print("\nCategorical Features:")
print(", ".join(categorical_features))

# Show missing values in the selected input features.
feature_summary = pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing_count": X_raw.isna().sum(),
    "missing_percent": X_raw.isna().mean().mul(100).round(2),
    "unique_values": X_raw.nunique(dropna=True)
})

display(feature_summary)


Numeric Features:
date_registration, average_CA_score, total_clicks

Categorical Features:
gender, region, highest_education, imd_band, age_band, studied_credits, disability


,dtype,missing_count,missing_percent,unique_values
date_registration,int64,0,0.00,156
average_CA_score,float64,0,0.00,1022
total_clicks,int64,0,0.00,715
gender,object,0,0.00,2
region,object,0,0.00,13
highest_education,object,0,0.00,5
imd_band,object,8,0.55,10
age_band,object,0,0.00,3
studied_credits,int64,0,0.00,11
disability,object,0,0.00,2


## Common: Split Data

Use the same train/test split for ES, LR, and SVM.


In [6]:
# Create one shared split for all systems.
# test_size=0.2 means 80% training and 20% testing.
# stratify=y keeps the class distribution similar in train and test sets.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Summarize train/test sizes and class proportions.
split_summary = pd.DataFrame({
    "set": ["train", "test"],
    "rows": [len(X_train_raw), len(X_test_raw)],
    "at_risk_percent": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

display(split_summary)


,set,rows,at_risk_percent
0,train,1161,33.94
1,test,291,34.02


## ES: Build Rule-Generation Preprocessor

Create the preprocessing pipeline used by the DT rule generator and ES.

Numeric features are median-imputed if needed, but not scaled.  
Categorical features are filled with `Unknown` and one-hot encoded.

The ES keeps unscaled numeric features so rule thresholds remain interpretable.


In [7]:
# OneHotEncoder handles categorical columns whose values are labels, not quantities.
# Example: region and highest_education are names/categories, not numeric measurements.
# One-hot encoding avoids creating a fake numeric order between categories.

try:
    es_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    es_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Numeric preprocessing for ES:
# Median imputation is a safe fallback if a numeric feature has missing values.
# No scaling is applied because ES rules should use original units.
es_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing for ES:
# Missing categorical values, especially imd_band, are kept as an explicit Unknown category.
es_categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", es_encoder)
])

# ES preprocessor used for DT rule generation and Experta inference.
es_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", es_numeric_pipeline, numeric_features),
        ("categorical", es_categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


## ES: Apply Rule-Generation Preprocessing

Fit the ES preprocessor on the training set and apply it to both training and test sets.


In [8]:
# Fit the ES preprocessor only on training data.
# This avoids using information from the test set during preprocessing.
X_train_processed_array = es_preprocessor.fit_transform(X_train_raw)

# Apply the already-fitted preprocessing to the test set.
X_test_processed_array = es_preprocessor.transform(X_test_raw)

# Get the column names after imputation and one-hot encoding.
processed_feature_names = es_preprocessor.get_feature_names_out().tolist()


## ES: Build Processed Dataframes

Store the ES processed arrays as dataframes.

These dataframes are used by the DT rule generator and the ES.


In [9]:
# Convert processed training array to dataframe.
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names,
    index=X_train_raw.index
)

# Convert processed test array to dataframe.
X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names,
    index=X_test_raw.index
)

# The number of rows should stay the same.
# The number of columns usually increases because of one-hot encoding.
print(f"Raw Training Shape: {X_train_raw.shape}")
print(f"Processed Training Shape: {X_train_processed.shape}")

X_train_processed.head()


Raw Training Shape: (1161, 10)
Processed Training Shape: (1161, 49)


,date_registration,average_CA_score,total_clicks,gender_Female,gender_Male,region_East Anglian Region,region_East Midlands Region,region_Ireland,region_London Region,region_North Region,region_North Western Region,region_Scotland,region_South East Region,region_South Region,region_South West Region,region_Wales,region_West Midlands Region,region_Yorkshire Region,highest_education_A Level or Equivalent,highest_education_HE Qualification,highest_education_Lower Than A Level,highest_education_No Formal quals,highest_education_Post Graduate Qualification,imd_band_0-10%,imd_band_10-20%,imd_band_20-30%,imd_band_30-40%,imd_band_40-50%,imd_band_50-60%,imd_band_60-70%,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%,imd_band_Unknown,age_band_0-35,age_band_35-55,age_band_55<=,studied_credits_60,studied_credits_75,studied_credits_90,studied_credits_120,studied_credits_135,studied_credits_150,studied_credits_180,studied_credits_210,studied_credits_220,studied_credits_240,disability_N,disability_Y
859,-121.0,65.604651,211.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1290,-31.0,71.046512,268.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
380,-81.0,82.279070,201.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1100,-32.0,12.674419,155.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
645,-44.0,78.395349,83.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## DT: Train Rule Generator

Train a small entropy-based DT.

The DT is used only to generate rules for the ES.


In [10]:
# Entropy makes the DT choose splits based on information gain.
# max_depth controls rule length.
# min_samples_split and min_samples_leaf prevent overly specific rules.
# class_weight="balanced" gives more training weight to the smaller class.
# This helps the model avoid ignoring At Risk students when the dataset has more Not At Risk examples.
dt_rule_generator = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Train the DT rule generator on the shared processed training data.
dt_rule_generator.fit(X_train_processed, y_train)

# Each leaf becomes one generated rule.
print(f"DT Depth: {dt_rule_generator.get_depth()}")
print(f"Generated Rules: {dt_rule_generator.get_n_leaves()}")


DT Depth: 4
Generated Rules: 14


## KB: Map Encoded Features

Map one-hot encoded feature names back to their original categorical meaning.


In [11]:
# The categorical transformer is a pipeline:
# imputer -> one-hot encoder.
# We retrieve the fitted encoder so we can recover the learned categories.
fitted_encoder = es_preprocessor.named_transformers_["categorical"].named_steps["encoder"]

# Get the exact one-hot feature names produced by the encoder.
encoded_categorical_feature_names = fitted_encoder.get_feature_names_out(categorical_features)

# This dictionary maps encoded feature names back to readable rule text.
# Example:
# imd_band_Unknown -> original_column = imd_band, category = Unknown
one_hot_feature_map = {}

encoded_index = 0

for original_column, categories in zip(categorical_features, fitted_encoder.categories_):
    for category in categories:
        encoded_name = encoded_categorical_feature_names[encoded_index]
        encoded_index += 1

        one_hot_feature_map[encoded_name] = {
            "original_column": original_column,
            "category": str(category)
        }


## KB: Format Conditions

Define a helper function for converting DT conditions into readable text.


In [12]:
def readable_condition(feature, operator, threshold):
    """Convert a DT condition into readable rule text."""

    # One-hot encoded categorical columns usually split at threshold 0.5.
    # Example:
    # schoolsup_yes > 0.5 means schoolsup is yes.
    # schoolsup_yes <= 0.5 means schoolsup is not yes.
    if feature in one_hot_feature_map and abs(threshold - 0.5) < 1e-8:
        original_column = one_hot_feature_map[feature]["original_column"]
        category = one_hot_feature_map[feature]["category"]

        if operator == ">":
            return f"{original_column} is {category}"

        return f"{original_column} is not {category}"

    # Numeric feature split.
    # Round the threshold text to keep rules readable.
    threshold_text = f"{threshold:.3f}".rstrip("0").rstrip(".")
    return f"{feature} {operator} {threshold_text}"


## KB: Define Rule Extraction Helpers

Create helper functions that convert DT leaf paths into IF–THEN rules.


In [13]:
def create_rule_from_leaf(tree_, node_id, conditions, class_names):
    """Create one rule from one DT leaf node."""

    # Class distribution at the leaf.
    # Example: counts for [Not At Risk, At Risk].
    class_counts = tree_.value[node_id][0]

    # The predicted class is the class with the largest count at this leaf.
    predicted_class_code = int(np.argmax(class_counts))

    # Confidence is the proportion of the predicted class at this leaf.
    total_weighted_samples = float(class_counts.sum())
    confidence = 0.0

    if total_weighted_samples > 0:
        confidence = float(class_counts[predicted_class_code] / total_weighted_samples)

    # Store all information needed for inference and reporting.
    return {
        # Raw list of rule conditions.
        # Each condition stores the feature, operator, threshold, and readable text.
        "conditions": conditions,

        # Human-readable version of the full rule path.
        # Conditions inside one DT path are joined with AND.
        "conditions_text": " AND ".join(condition["text"] for condition in conditions),

        # Numeric predicted class from the leaf.
        # 0 = Not At Risk, 1 = At Risk.
        "prediction_code": predicted_class_code,

        # Readable class label using the numeric prediction.
        "prediction_label": class_names[predicted_class_code],

        # Rule confidence based on the class distribution at this leaf.
        "confidence": round(confidence, 4),

        # Number of training samples that reached this leaf.
        # This helps show how general or specific the rule is.
        "training_samples": int(tree_.n_node_samples[node_id])
    }


def traverse_tree(tree_, node_id, conditions, feature_names, class_names, rules):
    """Traverse the DT and collect one rule per leaf."""

    # The feature used for splitting at the current node.
    feature_index = tree_.feature[node_id]

    # If this is not a leaf, continue traversing both branches.
    if feature_index != _tree.TREE_UNDEFINED:
        feature = feature_names[feature_index]
        threshold = float(tree_.threshold[node_id])

        # Left branch condition: feature <= threshold.
        left_condition = {
            "feature": feature,
            "operator": "<=",
            "threshold": threshold,
            "text": readable_condition(feature, "<=", threshold)
        }

        # Right branch condition: feature > threshold.
        right_condition = {
            "feature": feature,
            "operator": ">",
            "threshold": threshold,
            "text": readable_condition(feature, ">", threshold)
        }

        # Traverse the left child and add the left condition to the path.
        traverse_tree(
            tree_,
            tree_.children_left[node_id],
            conditions + [left_condition],
            feature_names,
            class_names,
            rules
        )

        # Traverse the right child and add the right condition to the path.
        traverse_tree(
            tree_,
            tree_.children_right[node_id],
            conditions + [right_condition],
            feature_names,
            class_names,
            rules
        )

        return

    # If this is a leaf, convert the current path into one rule.
    rules.append(
        create_rule_from_leaf(
            tree_=tree_,
            node_id=node_id,
            conditions=conditions,
            class_names=class_names
        )
    )


def extract_rules_from_tree(tree_model, feature_names, class_names):
    """Extract all DT root-to-leaf paths as KB rules."""

    # This list will contain all extracted rules.
    rules = []

    # Start traversal from node 0, which is the root of the tree.
    traverse_tree(
        tree_=tree_model.tree_,
        node_id=0,
        conditions=[],
        feature_names=feature_names,
        class_names=class_names,
        rules=rules
    )

    # Add stable rule IDs after extraction.
    for rule_id, rule in enumerate(rules, start=1):
        rule["rule_id"] = rule_id

    return rules


## KB: Generate Knowledge Base

Extract rules from the DT and display them as the ES knowledge base.


In [14]:
# Extract generated rules from the trained DT.
knowledge_base = extract_rules_from_tree(
    tree_model=dt_rule_generator,
    feature_names=processed_feature_names,
    class_names=class_names
)

# Convert the KB to a readable dataframe.
kb_table = pd.DataFrame([
    {
        "rule_id": rule["rule_id"],
        "conditions": rule["conditions_text"],
        "prediction": rule["prediction_label"],
        "confidence": rule["confidence"],
        "training_samples": rule["training_samples"]
    }
    for rule in knowledge_base
])

display(kb_table)


,rule_id,conditions,prediction,confidence,training_samples
0,1,average_CA_score <= 54.105 AND average_CA_score <= 33.395 AND average_CA_score <= 24.628,At Risk,1.0000,111
1,2,average_CA_score <= 54.105 AND average_CA_score <= 33.395 AND average_CA_score > 24.628 AND average_CA_score <= 29.174,At Risk,0.9146,13
2,3,average_CA_score <= 54.105 AND average_CA_score <= 33.395 AND average_CA_score > 24.628 AND average_CA_score > 29.174,At Risk,1.0000,13
3,4,average_CA_score <= 54.105 AND average_CA_score > 33.395 AND average_CA_score <= 44.616 AND total_clicks <= 54.5,At Risk,1.0000,16
4,5,average_CA_score <= 54.105 AND average_CA_score > 33.395 AND average_CA_score <= 44.616 AND total_clicks > 54.5,At Risk,0.8648,90
5,6,average_CA_score <= 54.105 AND average_CA_score > 33.395 AND average_CA_score > 44.616 AND date_registration <= -20.5,At Risk,0.7554,75
6,7,average_CA_score <= 54.105 AND average_CA_score > 33.395 AND average_CA_score > 44.616 AND date_registration > -20.5,Not At Risk,1.0000,6
7,8,average_CA_score > 54.105 AND average_CA_score <= 75.105 AND imd_band is not 90-100% AND average_CA_score <= 61.988,At Risk,0.5553,87
8,9,average_CA_score > 54.105 AND average_CA_score <= 75.105 AND imd_band is not 90-100% AND average_CA_score > 61.988,Not At Risk,0.6632,348
9,10,average_CA_score > 54.105 AND average_CA_score <= 75.105 AND imd_band is 90-100%,Not At Risk,1.0000,23


## KB: Export Knowledge Base

Save generated rules for reporting and reuse.


In [15]:
# Save readable KB table for the report.
kb_table.to_csv("oulad_generated_expert_system_rules.csv", index=False)

# Save full KB structure for reuse in code.
with open("oulad_generated_expert_system_knowledge_base.json", "w", encoding="utf-8") as file:
    json.dump(knowledge_base, file, indent=2)

print("Saved oulad_generated_expert_system_rules.csv")
print("Saved oulad_generated_expert_system_knowledge_base.json")


Saved oulad_generated_expert_system_rules.csv
Saved oulad_generated_expert_system_knowledge_base.json


## ES: Import Experta

Import Experta for rule-based inference.


In [16]:
# Experta may depend on old collections names.
# This patch maps old names to their current locations.
for name in ["Mapping", "MutableMapping", "Sequence"]:
    if not hasattr(collections, name):
        setattr(collections, name, getattr(collections.abc, name))

# Try importing Experta.
# If missing, install it and import again.
try:
    from experta import KnowledgeEngine, Fact, Rule, AS
except ModuleNotFoundError:
    print("Experta is not installed. Installing experta...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "experta"])
    from experta import KnowledgeEngine, Fact, Rule, AS

print("Experta imported successfully.")


Experta imported successfully.


## ES: Define Inference Helpers

Define how the ES checks whether a KB rule matches a student.


In [17]:
class Student(Fact):
    """One processed student record."""
    pass


def condition_matches(value, operator, threshold):
    """Evaluate one extracted KB condition for the current student."""

    # The DT learned these conditions, but after extraction they are stored as plain data.
    # The ES needs to interpret the stored operator and threshold during inference.
    if operator == "<=":
        return value <= threshold

    if operator == ">":
        return value > threshold

    raise ValueError(f"Unsupported operator: {operator}")


def rule_matches_student(student_features, rule):
    """Check whether all conditions in one rule match a student."""

    # Conditions inside one rule are AND conditions.
    # Different rules are OR alternatives.
    for condition in rule["conditions"]:
        feature = condition["feature"]
        operator = condition["operator"]
        threshold = condition["threshold"]

        # Get the current student's value for the feature used in this rule condition.
        # Numeric features contain values such as average_CA_score or total_clicks.
        # One-hot encoded categorical features contain 0 or 1.
        value = float(student_features.get(feature, 0))

        if not condition_matches(value, operator, threshold):
            return False

    return True


## ES: Define Engine

Define the Experta engine that applies KB rules.


In [18]:
class AcademicRiskExpertSystem(KnowledgeEngine):
    """Experta engine for academic risk prediction."""

    def __init__(self, rules):
        super().__init__()

        # Store generated KB rules.
        self.rules = rules

        # Store the output after inference.
        self.result = None

    # This Experta rule fires when a Student fact is added to the engine.
    # AS.student captures the matched Student fact and passes it to the function.
    @Rule(AS.student << Student())
    def infer_academic_risk(self, student):
        """Infer risk class for one student."""

        # Convert Experta fact into a normal dictionary.
        student_features = {
            key: value
            for key, value in dict(student).items()
            if key in processed_feature_names
        }

        # Find all KB rules that match this student.
        matching_rules = [
            rule for rule in self.rules
            if rule_matches_student(student_features, rule)
        ]

        # Safety fallback.
        # A complete DT-derived KB should normally match exactly one rule.
        if not matching_rules:
            self.result = {
                "prediction_code": None,
                "prediction_label": "Unknown",
                "rule_id": None,
                "confidence": 0.0,
                "explanation": "No rule matched this student."
            }
            return

        # Each DT-generated rule represents one complete root-to-leaf path.
        # For one student, only one path should match because each DT split is either <= threshold or > threshold.
        # Therefore, the matching list should contain exactly one rule.
        selected_rule = matching_rules[0]

        # Store final ES output.
        self.result = {
            "prediction_code": selected_rule["prediction_code"],
            "prediction_label": selected_rule["prediction_label"],
            "rule_id": selected_rule["rule_id"],
            "confidence": selected_rule["confidence"],
            "explanation": selected_rule["conditions_text"]
        }


## ES: Define Prediction Function

Run ES inference for one processed student.


In [19]:
def infer_processed_student(processed_student_row, rules=knowledge_base):
    """Run ES inference for one processed test row."""

    # Create a fresh ES engine so this prediction is independent.
    engine = AcademicRiskExpertSystem(rules)

    # Reset the engine to make sure it starts with no previous facts.
    engine.reset()

    # Convert the processed student row from a pandas Series into a dictionary.
    student_features = processed_student_row.to_dict()

    # Create an Experta Student fact from the processed feature values.
    student_fact = Student(**student_features)

    # Add the Student fact to the ES engine.
    engine.declare(student_fact)

    # Run the ES inference process.
    engine.run()

    # Return the prediction produced by the matching rule.
    return engine.result

## ML: Build LR and SVM Pipelines

Build Logistic Regression and SVM using model-appropriate preprocessing.

LR and SVM use the same selected features and train/test split as the ES, but they also scale numeric features because they are sensitive to feature scale.


In [20]:
def make_ml_preprocessor():
    """Create preprocessing for LR and SVM."""

    try:
        ml_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ml_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    # Numeric preprocessing for ML:
    # Median imputation handles missing numeric values.
    # StandardScaler puts numeric features on a comparable scale.
    ml_numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing for ML:
    # Missing categories are kept as Unknown, then one-hot encoded.
    ml_categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", ml_encoder)
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", ml_numeric_pipeline, numeric_features),
            ("categorical", ml_categorical_pipeline, categorical_features)
        ],
        remainder="drop"
    )


lr_model = Pipeline(steps=[
    ("preprocessor", make_ml_preprocessor()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

svm_model = Pipeline(steps=[
    ("preprocessor", make_ml_preprocessor()),
    ("classifier", SVC(
        kernel="rbf",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

print("LR and SVM pipelines created.")


LR and SVM pipelines created.


## ML: Train LR and SVM

Train LR and SVM on the raw training dataframe.

The preprocessing is applied inside each pipeline.


In [21]:
# Train LR with ML-specific preprocessing.
lr_model.fit(X_train_raw, y_train)

# Train SVM with ML-specific preprocessing.
svm_model.fit(X_train_raw, y_train)

print("LR trained.")
print("SVM trained.")


LR trained.
SVM trained.


## ES: Generate Predictions

Generate ES predictions on the processed test set.


In [22]:
# Run ES inference for every processed test row.
es_results = [
    infer_processed_student(row)
    for _, row in X_test_processed.iterrows()
]

# Extract numeric prediction codes from ES results.
es_predictions = np.array([
    result["prediction_code"]
    for result in es_results
])


## LR and SVM: Generate Predictions

Generate LR and SVM predictions on the raw test dataframe.

The fitted ML pipelines apply imputation, encoding, and scaling internally.


In [23]:
# Predict test labels using LR.
lr_predictions = lr_model.predict(X_test_raw)

# Predict test labels using SVM.
svm_predictions = svm_model.predict(X_test_raw)

# Store all predictions in one dictionary for shared evaluation.
predictions = {
    "ES": es_predictions,
    "LR": lr_predictions,
    "SVM": svm_predictions
}


## Common: Evaluation Metrics

Calculate test-set metrics for ES, LR, and SVM.


In [24]:
def calculate_metrics(y_true, y_pred):
    """Calculate classification metrics for the At Risk class."""

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_true,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        )
    }


# Build one comparison row per system.
metrics_table = pd.DataFrame([
    {"System": system_name, **calculate_metrics(y_test, y_pred)}
    for system_name, y_pred in predictions.items()
])

# Round metrics for cleaner display.
for column in ["Accuracy", "Precision", "Recall", "F1-score"]:
    metrics_table[column] = metrics_table[column].round(4)

print("Positive Class: At Risk")
display(metrics_table)


Positive Class: At Risk


,System,Accuracy,Precision,Recall,F1-score
0,ES,0.8316,0.7404,0.7778,0.7586
1,LR,0.8282,0.7379,0.7677,0.7525
2,SVM,0.8488,0.7835,0.7677,0.7755


## Common: Detailed Reports

Display clean test-set metrics and confusion matrices for all systems.


In [25]:
def build_confusion_matrix_table(y_true, y_pred):
    """Return a readable confusion matrix table."""

    # Labels are ordered as:
    # 0 = Not At Risk
    # 1 = At Risk
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return pd.DataFrame(
        cm,
        index=["Actual: Not At Risk", "Actual: At Risk"],
        columns=["Predicted: Not At Risk", "Predicted: At Risk"]
    )


def build_classification_metrics(y_true, y_pred):
    """Return binary classification metrics for the positive class."""

    return {
        # Overall percentage of correct predictions.
        "Accuracy": round(
            accuracy_score(y_true, y_pred),
            4
        ),

        # Precision for class 1.
        "Precision": round(
            precision_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        ),

        # Recall for class 1.
        "Recall": round(
            recall_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        ),

        # F1-score for class 1.
        # This is binary F1, not macro, micro, or weighted F1.
        "F1-score": round(
            f1_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        )
    }


for system_name, y_pred in predictions.items():
    print(f"\n{system_name} - Positive Class: At Risk")

    metrics_table = pd.DataFrame(
        [build_classification_metrics(y_test, y_pred)]
    )

    display(metrics_table)

    confusion_matrix_table = build_confusion_matrix_table(y_test, y_pred)

    display(confusion_matrix_table)


ES - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.8316,0.7404,0.7778,0.7586


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,165,27
Actual: At Risk,22,77



LR - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.8282,0.7379,0.7677,0.7525


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,165,27
Actual: At Risk,23,76



SVM - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.8488,0.7835,0.7677,0.7755


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,171,21
Actual: At Risk,23,76


## Summary

The notebook compares:

- **ES:** Experta expert system using DT-generated rules from unscaled, interpretable features
- **LR:** Logistic Regression with numeric scaling
- **SVM:** Support Vector Machine with numeric scaling

All systems use the processed OULAD dataset, the same selected features, the same train/test split, and the same test-set metrics.
